# garimpo-uncertainty on Google Colab

This notebook reproduces the experiment on Google Colab, in the order of a training notebook:
setup, data, run configs, tests, training of the four models, evaluation, and the release of the
GPU. It downloads nothing: the Sentinel-2 data must already be in Drive as
`MyDrive/garimpo/data.tar.zst`.

## How to use it

1. Open the notebook as "Open the notebook" says, on a GPU runtime: Runtime > Change runtime type.
   Changing the runtime later starts a new session, which erases the disk.
2. Section 3 chooses the scale. `pilot` trains the four models on 1% of the training tiles for a
   few updates, to check the code; `full` trains them on 100%, the experiment itself. Left at
   `auto`, it continues the full runs once one of them has a checkpoint in Drive, and runs the
   pilot otherwise; choose `full` once, after the pilot passes.
3. Run Runtime > Run all and keep the tab open. The last cell writes the pending files to Drive
   and releases the GPU.
4. If the session ends before the last cell, open a new session and run Runtime > Run all again.
   Finished stages are skipped, and a training continues from its last checkpoint.

The title of a cell that is not part of every run says so:

- "- (optional)": it runs unless you clear its `RUN` box.
- "- (in a reexecution)": it does nothing unless you check its `RUN` box. Use it after a code fix,
  to redo a run from a given stage.

| Section | What it does | Saved in Drive |
|---|---|---|
| 1. Setup | Mounts Drive, clones the code, builds the Python environment | Nothing |
| 2. Data | Unpacks the data and restores the earlier runs and logs | Nothing |
| 3. Run configs | Chooses the pilot or the full runs, and shows their values | Nothing |
| 4. Tests | Runs the test suite on the GPU | Nothing |
| 5. Training | Runs the four models, each through every stage | `runs/`, `logs/` |
| 6. Evaluation | Merges the four runs' result tables | `outputs/` for the full runs |
| 7. Release | Writes the pending files to Drive and releases the GPU | Everything above |

## Open the notebook

Open the notebook from its GitHub link,
<https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/colab.ipynb>,
in every session, so the cells are always those on `main`. A copy made with File > Upload notebook
or File > Save a copy in Drive keeps the cells of the day it was made, and the clone does not update
them. Options changed in a form are not saved either: they stay set in the open tab, and reopening
the notebook from the link restores the values on `main`, where `FORCE_FROM` and `FORCE_VALIDATE`
are empty.

The session clones the repository without credentials and cannot push, so change the code on your
own machine and push it from there. The next session clones the new version.

## 1. Setup

The runtime's disk is erased when a session ends. These cells clone the code again and build the
Python environment; section 2 copies the data and the earlier runs back from Drive.

Mount Google Drive. Colab asks for permission the first time. Everything that must outlive the
session is kept in `MyDrive/garimpo`.

In [ ]:
# @title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

The paths and variables the other cells use. `DATA` must stay `/content/garimpo-data`: every run
records the path of its data and refuses to continue when it is restored under another path.

In [ ]:
# @title Settings
import os
import subprocess
import threading
import time
from pathlib import Path

from google.colab import runtime

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions
DATA = "/content/garimpo-data"  # runs record this path, so it never changes

os.environ["GARIMPO_DATA_DIR"] = DATA
periodic_copy = None  # the copy to Drive every 30 minutes of section 5, once started

The functions the other cells use. Every pilot and full-run job ends by printing `PIPELINE CHECK: PASS`
or `PIPELINE CHECK: FAIL`, and a cell keeps going when one of its shell commands fails, so read
that line to know how a job went. A job's log keeps every attempt: `start_log` marks where a new
one begins, and `print_verdicts` prints the verdict of the last attempt only.
`copy_to_drive_every_30_min` saves the runs and logs during the training. `release_runtime` ends the
notebook: it writes the files still pending to Drive, then releases the machine. When a Python
error stops a cell, Colab cancels the cells queued after it, release included. No cell before the
release raises one: a failed shell command only prints its error, and the Python cells that can
fail print theirs. If you interrupt a cell yourself, run the cells of section 7 by hand.

In [ ]:
# @title Define the helper functions
VERDICTS = ("PIPELINE CHECK: PASS", "PIPELINE CHECK: FAIL")
ATTEMPT = "=== attempt started "


def start_log(name: str) -> None:
    """Mark the start of a new attempt in the job's log. It never raises."""
    started = time.strftime("%Y-%m-%d %H:%M", time.gmtime())
    try:
        with (Path("outputs/logs") / f"{name}.log").open("a", encoding="utf-8") as log:
            log.write(f"{ATTEMPT}{started} UTC ===\n")
    except OSError as error:
        print(f"Could not mark the attempt in the log: {error}")


def print_verdicts(log_names: list[str]) -> None:
    """Print the pipeline-check line of the last attempt in each log. It never raises."""
    for name in log_names:
        log = Path("outputs/logs") / f"{name}.log"
        try:
            text = log.read_text(encoding="utf-8", errors="replace")
        except OSError as error:
            print(name, f"NO LOG: {error}")
            continue
        text = text.rsplit(ATTEMPT, 1)[-1]
        found = [line for line in text.splitlines() if line.startswith(VERDICTS)]
        print(name, found[-1] if found else "NO VERDICT: the job or its check died; see the log")


def copy_to_drive_every_30_min(stop: threading.Event) -> None:
    """Copy runs/ and the logs to Drive every 30 minutes until stop is set."""
    runs = ["rsync", "-a", "--exclude", "*.tmp", "runs/", f"{DRIVE}/runs/"]
    logs = ["rsync", "-a", "outputs/logs/", f"{DRIVE}/logs/"]
    while not stop.wait(1800):
        for command in (runs, logs):
            try:
                copied = subprocess.run(command, check=False)  # noqa: S603
            except OSError as error:
                print(f"The copy to Drive every 30 minutes failed: {error}")
                continue
            if copied.returncode != 0:
                print(f"The copy to Drive every 30 minutes ended with code {copied.returncode}")


def release_runtime() -> None:
    """Write pending files to Drive, then release the runtime, which ends the session."""
    try:
        drive.flush_and_unmount()
    except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
        print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()

`rsync` copies the runs to and from Drive, and `zstd` unpacks the data archive. The
pipeline compares each input file's size and modification time to tell which stages are already
done, and both `rsync -a` and `tar` keep those times.

In [ ]:
# @title Install rsync and zstd
!apt-get -qq install -y rsync zstd > /dev/null

`uv` builds the Python environment from `uv.lock`, which pins every package version.

In [ ]:
# @title Install uv
!pip install -q uv

Clone the code with its `compass/` submodule, the adapted COMPASS method. A session starts in
`/content`, so git clones into `/content/garimpo-uncertainty`, the `CODE` folder the next cell
enters. Run it once per session: run again after that cell, it clones a second copy inside `CODE`.

In [ ]:
# @title Clone the code
!git clone --recurse-submodules {REPO}

In [ ]:
# @title Enter the code folder
%cd {CODE}

In [ ]:
# @title Build the Python environment
!uv sync

The commit this session runs. Note it next to any result you keep.

In [ ]:
# @title Show the commit
!git log -1 --format='%h %cs %s'

In [ ]:
# @title Create the folders
!mkdir -p {DRIVE}/runs {DRIVE}/logs {DATA} runs outputs/logs

## 2. Data

Unpack the data archive, `MyDrive/garimpo/data.tar.zst`, which must already be in Drive. The
unpacked store always takes 69.2 GB, whatever the training fraction, because its arrays are created
for all 18,864 cells. `tar` lists each file as it unpacks it, and reports the archive missing if it
is not in Drive.

In [ ]:
# @title Unpack the data
!tar -I zstd -xvf {DRIVE}/data.tar.zst -C {DATA}

Copy back the runs and logs of earlier sessions. The finished stages of a run are then skipped,
and each job's log keeps the earlier attempts above the new one, including one cut short by a
session that ended. If one of these two cells
fails, stop Run all with Runtime > Interrupt execution and fix it first: a training that runs
without its earlier run starts over, and its copy to Drive replaces the saved one.

In [ ]:
# @title Restore the runs
!rsync -a {DRIVE}/runs/ runs/

In [ ]:
# @title Restore the logs
!rsync -a {DRIVE}/logs/ outputs/logs/

Free disk. The store needs 69.2 GB and the Python environment 6.9 GB, plus room for the runs.

In [ ]:
# @title Show the free disk
!df -h /content

## 3. Run configs

Each run reads every value from its JSON file, versioned with the code: `configs/full/` holds the
four models on 100% of the training tiles, the experiment itself, and `configs/pilot/` the same
four models on 1%. A pilot config differs from its full one only in its scale: 1% of the training
tiles, 2 epochs of 20 updates, and every layer accepted, so that a model trained for a few updates
still reaches the last stage. The pilot checks on the GPU the code the full runs will run; its
numbers say nothing about the methods, and its result tables go to `runs/pilot/tables/`, never to
`outputs/`.

To change a value, edit the file on your own machine, commit and push it; the next session clones
it. A job refuses a config with a value missing, empty or misspelled before it runs any stage.

`SCALE` chooses which configs the training reads. At `auto`, the cell looks for the `last.pt` that a
run saves after its first epoch, in the runs restored from Drive: if any full run has one, it
continues the full runs, and otherwise it runs the pilot. It prints what it found, so check that
line before you leave Run all going. Choose `pilot` or `full` to override it. A full run's first
checkpoint reaches Drive with the first copy, 30 minutes into the training: if a session ends
before that, choose `full` in the next one.

In [ ]:
# @title Choose the pilot or the full runs
import json

SCALE = "auto"  # @param ["auto", "pilot", "full"]
MODELS = ["unet_scratch", "unet_imagenet", "unet_sentinel2", "convnet"]


def checkpointed(scale: str) -> list[str]:
    """Return the models whose run of this scale saved a checkpoint. It never raises."""
    found = []
    for model in MODELS:
        path = Path("configs") / scale / f"{model}.json"
        try:
            run_dir = Path(json.loads(path.read_text(encoding="utf-8"))["RUN_DIR"])
        except (OSError, ValueError, KeyError) as error:
            print(f"Could not read the RUN_DIR of {path}: {error!r}")
            continue
        if (run_dir / "last.pt").exists():
            found.append(model)
    return found


full_runs, pilot_runs = checkpointed("full"), checkpointed("pilot")
if SCALE == "auto":
    SCALE = "full" if full_runs else "pilot"
print(f"Training the {SCALE} runs.")
print("Full runs with a checkpoint, which continue:", ", ".join(full_runs) or "none")
print("Pilot runs with a checkpoint:", ", ".join(pilot_runs) or "none")

In [ ]:
# @title Show the run configs
import pandas as pd

try:
    configs = {
        model: json.loads((Path("configs") / SCALE / f"{model}.json").read_text(encoding="utf-8"))
        for model in MODELS
    }
    RUN_DIRS = " ".join(config["RUN_DIR"] for config in configs.values())
    display(pd.DataFrame(configs))
except (OSError, ValueError, KeyError) as error:
    # A raise here would cancel the release queued at the end of the notebook.
    RUN_DIRS = ""
    print(f"Could not read the {SCALE} configs: {error!r}")

## 4. Tests

Optional after the first session: clear the `RUN` box of the cell to skip them. The tests build a
small synthetic dataset of their own. GitHub's CI runs them on a CPU; here they run on CUDA. A few
tests skip on a GPU on purpose, such as those that compare a result with one computed on the CPU,
and every other test should pass. A failure does not stop the cells after it, so read the output
of this cell before you trust the training.

In [ ]:
# @title Run the tests on the GPU - (optional)
RUN = True  # @param {type:"boolean"}
if RUN:
    !GARIMPO_TEST_DEVICE=cuda uv run --no-sync pytest -q

## 5. Training

The four models all use a ResNet-18 encoder, each from its config in `configs/<scale>/`. Each run
trains, then goes through every stage, chooses the COMPASS-J layer on the validation set and writes
the reason to `chosen_layer.json`. Each one writes its `RUN_DIR` and the log
`outputs/logs/<scale>-<model>.log`. A full training may not fit in one session: run the notebook
again in a new one, and each run resumes where it stopped.

Copy `runs/` and the logs to Drive every 30 minutes until section 7, so a session that ends early
keeps the finished stages, the last checkpoint and the log up to that point. Running this cell
again while the copy runs starts nothing new.

In [ ]:
# @title Start the copy to Drive every 30 minutes
if periodic_copy is None or not periodic_copy.is_alive():
    stop_copy = threading.Event()
    periodic_copy = threading.Thread(
        target=copy_to_drive_every_30_min, args=(stop_copy,), daemon=True
    )
    periodic_copy.start()

In [ ]:
# @title Train the U-Net from scratch
start_log(f"{SCALE}-unet_scratch")
!NUM_WORKERS={os.cpu_count()} bash slurm/run.sbatch configs/{SCALE}/unet_scratch.json 2>&1 | tee -a outputs/logs/{SCALE}-unet_scratch.log

In [ ]:
# @title Train the U-Net from ImageNet weights
start_log(f"{SCALE}-unet_imagenet")
!NUM_WORKERS={os.cpu_count()} bash slurm/run.sbatch configs/{SCALE}/unet_imagenet.json 2>&1 | tee -a outputs/logs/{SCALE}-unet_imagenet.log

In [ ]:
# @title Train the U-Net from Sentinel-2 MoCo weights
start_log(f"{SCALE}-unet_sentinel2")
!NUM_WORKERS={os.cpu_count()} bash slurm/run.sbatch configs/{SCALE}/unet_sentinel2.json 2>&1 | tee -a outputs/logs/{SCALE}-unet_sentinel2.log

In [ ]:
# @title Train the ConvNet (DeepLabV3)
start_log(f"{SCALE}-convnet")
!NUM_WORKERS={os.cpu_count()} bash slurm/run.sbatch configs/{SCALE}/convnet.json 2>&1 | tee -a outputs/logs/{SCALE}-convnet.log

After a code fix, push it, start a new session, check the `RUN` box of the model to redo, and set
`FORCE_FROM` to the first stage the fix touches: that stage and the ones after it run again, and
the earlier ones are skipped. Left empty, it runs what is not finished yet, as the cell above does.
`FORCE_FROM=train` refuses while `last.pt` holds a training that has not finished. `FORCE_VALIDATE`
checks the data again.

In [ ]:
# @title Rerun the U-Net from scratch - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log(f"{SCALE}-unet_scratch")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/run.sbatch configs/{SCALE}/unet_scratch.json 2>&1 | tee -a outputs/logs/{SCALE}-unet_scratch.log

In [ ]:
# @title Rerun the U-Net from ImageNet weights - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log(f"{SCALE}-unet_imagenet")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/run.sbatch configs/{SCALE}/unet_imagenet.json 2>&1 | tee -a outputs/logs/{SCALE}-unet_imagenet.log

In [ ]:
# @title Rerun the U-Net from Sentinel-2 MoCo weights - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log(f"{SCALE}-unet_sentinel2")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/run.sbatch configs/{SCALE}/unet_sentinel2.json 2>&1 | tee -a outputs/logs/{SCALE}-unet_sentinel2.log

In [ ]:
# @title Rerun the ConvNet (DeepLabV3) - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log(f"{SCALE}-convnet")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/run.sbatch configs/{SCALE}/convnet.json 2>&1 | tee -a outputs/logs/{SCALE}-convnet.log

One line per model. `PASS` means every stage finished and every check held. After `FAIL`, the
lines above it in the log name the stage that failed and its error. `NO VERDICT` means the job or
its check died first, and the log shows where. `NO LOG` means that model has no log yet, or
its log could not be read.

In [ ]:
# @title Print the verdicts
print_verdicts([f"{SCALE}-{model}" for model in MODELS])

## 6. Evaluation

The merge writes `results.csv` and the comparison tables from the four runs: to `outputs/` for the
full runs, to `runs/pilot/tables/` for the pilot. It needs all four to have passed. After the full
runs, download `MyDrive/garimpo/outputs/` and commit it from your own machine: these are the final
result tables.

In [ ]:
# @title Merge the four runs
TABLES = "runs/pilot/tables" if SCALE == "pilot" else "outputs"
!uv run --no-sync python scripts/merge_results.py --out {TABLES} --runs {RUN_DIRS}

In [ ]:
# @title Save the tables in Drive
!rsync -a outputs/ {DRIVE}/outputs/

## 7. Release

Stop the copy every 30 minutes, then copy the runs and logs to Drive one last time and release the
GPU. The stop cell waits for a copy in progress to finish, so that it and the final copy never write
to Drive at the same time.

In [ ]:
# @title Stop the copy every 30 minutes
if periodic_copy is not None:
    stop_copy.set()
    periodic_copy.join()

In [ ]:
# @title Save the runs in Drive
!rsync -a runs/ {DRIVE}/runs/

In [ ]:
# @title Save the logs in Drive
!rsync -a outputs/logs/ {DRIVE}/logs/

In [ ]:
# @title Write pending files to Drive and release the GPU
release_runtime()

## Where the files are

| Path | Contents | Kept after the session |
|---|---|---|
| `MyDrive/garimpo/data.tar.zst` | The data, which the notebook only reads | Yes |
| `MyDrive/garimpo/runs/` | Every run: checkpoints, tables, stage statuses | Yes |
| `MyDrive/garimpo/logs/` | The log of every pilot and full-run job | Yes |
| `MyDrive/garimpo/outputs/` | The merged tables of the full runs | Yes |
| `/content/garimpo-uncertainty/` | The code | No |
| `/content/garimpo-data/` | The unpacked data | No |